# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

In [4]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

### Data Preparation

In [5]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f194e2e2cd0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f194e2e1da0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f194e2e2e80>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f194e2e2bb0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:

In [7]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [8]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [7]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [8]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [9]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [14]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=128, shuffle=False, collate_fn=transformers.default_data_collator
)

import numpy as np
from tqdm.notebook import tqdm
total_correct = 0
total_samples = 0
model = model.to(device)
with torch.no_grad():
    for batch in tqdm(val_loader):
        predicted = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            token_type_ids=batch["token_type_ids"].to(device),
        )
        true_labels = batch['labels'].cpu().data.numpy()
        pred_labels = np.argmax(torch.softmax(predicted.logits, dim=1).data.cpu().numpy(), axis=-1)
        batch_accuracy = np.sum(true_labels == pred_labels)


        total_correct += batch_accuracy
        total_samples += len(batch['labels'])

accuracy = total_correct / total_samples

  0%|          | 0/316 [00:00<?, ?it/s]

In [15]:
accuracy

0.9083848627256987

In [16]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [1]:
import transformers
model_name = "microsoft/deberta-v3-base"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1a4960>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1a4570>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1a44e0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1539c0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1a51d0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e4d3db0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f7aa06195c0>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:154: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7f781e1a5890>
  self._event_pipes[threading.current_thread()] = event_pipe
/home/korenikil/NLP/nlp_course/.venv/lib/python3.11/site-packages/ipykernel/iostream.py:

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=f"./{model_name.split('/')[-1]}",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    logging_dir="./logs",
    report_to="none",
    dataloader_pin_memory=False,

    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=256,
    dataloader_num_workers=4
)



data_collator = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"]
)

In [11]:
trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 15.63 GiB of which 32.56 MiB is free. Process 3564912 has 290.00 MiB memory in use. Including non-PyTorch memory, this process has 15.30 GiB memory in use. Of the allocated memory 14.98 GiB is allocated by PyTorch, and 128.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>